# importing Libraries

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Data Loading & Exploration

In [3]:
path = ('diabetes.csv')
df = pd.read_csv(path)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.describe

<bound method NDFrame.describe of      Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0              6      148             72             35        0  33.6   
1              1       85             66             29        0  26.6   
2              8      183             64              0        0  23.3   
3              1       89             66             23       94  28.1   
4              0      137             40             35      168  43.1   
..           ...      ...            ...            ...      ...   ...   
763           10      101             76             48      180  32.9   
764            2      122             70             27        0  36.8   
765            5      121             72             23      112  26.2   
766            1      126             60              0        0  30.1   
767            1       93             70             31        0  30.4   

     DiabetesPedigreeFunction  Age  Outcome  
0                       0.627  

In [5]:
df.shape

(768, 9)

In [6]:
df.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

# Data Cleaning (Insulin, SkinThickness, BloodPressure, etc.)

In [7]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [8]:
cols_with_invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
(df[cols_with_invalid_zeros] == 0).sum()

Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

* calculating the percentage of zeros in Feature columns 

In [9]:
(df[cols_with_invalid_zeros] == 0).mean() * 100

Glucose           0.651042
BloodPressure     4.557292
SkinThickness    29.557292
Insulin          48.697917
BMI               1.432292
dtype: float64

In [10]:
df.groupby (df['Insulin'] == 0 ) ['Outcome'].mean()

Insulin
False    0.329949
True     0.368984
Name: Outcome, dtype: float64

In [11]:
print(f"Minimum Glucose (excluding zeros): {df[df['Glucose'] != 0]['Glucose'].min()}")
print(f"Maximum Glucose: {df['Glucose'].max()}")

Minimum Glucose (excluding zeros): 44
Maximum Glucose: 199


In [12]:
print(f"Mean of Glucose: {df['Glucose'].mean()}")
print(f"Standard Deviation of Glucose: {df['Glucose'].std()}")

Mean of Glucose: 120.89453125
Standard Deviation of Glucose: 31.97261819513622


In [13]:
df = df[df['Glucose'] != 0].copy()
display(df.head())

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [14]:
df['Glucose_bin'] = pd.cut(df['Glucose'], bins=[0, 100, 140, 200], labels=['Normal','Pre-Diabetes','High'], right=False)
df['Glucose_bin'].value_counts()

Glucose_bin
Pre-Diabetes    374
High            197
Normal          192
Name: count, dtype: int64

In [15]:
df[df['Insulin'] != 0].groupby('Glucose_bin') ['Insulin'].median()

Glucose_bin
Normal           66.0
Pre-Diabetes    130.0
High            194.0
Name: Insulin, dtype: float64

In [16]:
df['Insulin'] = df['Insulin'].replace(0, np.nan)
df['Insulin'] = df['Insulin'].fillna(df.groupby('Glucose_bin')['Insulin'].transform('median'))

In [17]:
(df['Insulin'] == 0).sum()

np.int64(0)

In [18]:
df = df[df['BMI'] != 0].copy()
display(df.head())

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Glucose_bin
0,6,148,72,35,194.0,33.6,0.627,50,1,High
1,1,85,66,29,66.0,26.6,0.351,31,0,Normal
2,8,183,64,0,194.0,23.3,0.672,32,1,High
3,1,89,66,23,94.0,28.1,0.167,21,0,Normal
4,0,137,40,35,168.0,43.1,2.288,33,1,Pre-Diabetes


In [19]:
df['BMI_bin'] = pd.cut(
    df['BMI'], bins=[0, 18.4, 24.9, 29.9, np.inf],
    labels= ['Underweight', 'Normal', 'Overweight', 'Obese'],
    right=True
    )

df['BMI_bin'].value_counts()

BMI_bin
Obese          469
Overweight     178
Normal         101
Underweight      4
Name: count, dtype: int64

In [20]:
df['SkinThickness'] = df['SkinThickness'].replace(0, np.nan)
df['SkinThickness'] = df['SkinThickness'].fillna(df.groupby('BMI_bin')['SkinThickness'].transform('median'))

In [21]:
(df['SkinThickness'] == 0).sum()

np.int64(0)

In [22]:
df['SkinThickness'].isna().sum()

np.int64(0)

In [23]:
df = df[df['BloodPressure'] != 0].copy()

In [24]:
(df['BloodPressure'] == 0).sum()

np.int64(0)

In [25]:
display(df.head())

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Glucose_bin,BMI_bin
0,6,148,72,35.0,194.0,33.6,0.627,50,1,High,Obese
1,1,85,66,29.0,66.0,26.6,0.351,31,0,Normal,Overweight
2,8,183,64,17.0,194.0,23.3,0.672,32,1,High,Normal
3,1,89,66,23.0,94.0,28.1,0.167,21,0,Normal,Overweight
4,0,137,40,35.0,168.0,43.1,2.288,33,1,Pre-Diabetes,Obese


# Step 3 — Train/Test Split

In [26]:
X = df[['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']]
y = df['Outcome']

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [28]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Outcome
0    0.656304
1    0.343696
Name: proportion, dtype: float64
Outcome
0    0.655172
1    0.344828
Name: proportion, dtype: float64


# Model Comparison

* Logistic Regression (BaseLine)

In [29]:
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

pipeline_lr.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pregnancies','Glucose','BloodPressure',...,'BMI', 'DiabetesPedigreeFunction','Age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [30]:
y_pred_lr = pipeline_lr.predict(X_test)
print(f"Classification Report:\n{classification_report(y_test, y_pred_lr)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.80      0.78        95
           1       0.58      0.52      0.55        50

    accuracy                           0.70       145
   macro avg       0.67      0.66      0.66       145
weighted avg       0.70      0.70      0.70       145



* Logistic Regression (Balanced)

In [31]:
pipeline_lr_bal = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced'))
])

pipeline_lr_bal.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pregnancies','Glucose','BloodPressure',...,'BMI', 'DiabetesPedigreeFunction','Age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [32]:
y_pred_lr_bal = pipeline_lr_bal.predict(X_test)

print(f"Classification Report:\n{classification_report(y_test, y_pred_lr_bal)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.69      0.74        95
           1       0.53      0.66      0.59        50

    accuracy                           0.68       145
   macro avg       0.66      0.68      0.67       145
weighted avg       0.70      0.68      0.69       145



* Training with Random Forest

* Ramdom Forest (BaseLine)

In [34]:
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier( random_state=42))
])

pipeline_rf.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pregnancies','Glucose','BloodPressure',...,'BMI', 'DiabetesPedigreeFunction','Age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [35]:
y_pred_rf = pipeline_rf.predict(X_test)

print(f"Classification Report:\n{classification_report(y_test, y_pred_rf)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.81      0.81        95
           1       0.64      0.64      0.64        50

    accuracy                           0.75       145
   macro avg       0.73      0.73      0.73       145
weighted avg       0.75      0.75      0.75       145



* Ramdom Forest (Balanced)

"Compared 4 model variants — see results above. Chose RF-balanced for its stronger precision-recall balance and interpretability, with only a small recall trade-off vs. LR-balanced."

In [36]:
pipeline_rf_bal = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])

pipeline_rf_bal.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pregnancies','Glucose','BloodPressure',...,'BMI', 'DiabetesPedigreeFunction','Age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [37]:
importances = pipeline_rf_bal.named_steps['classifier'].feature_importances_

In [38]:
print(importances)

[0.08015615 0.23916893 0.07067349 0.09185641 0.13171262 0.15500706
 0.10795381 0.12347153]


In [39]:
feature_names = X_train.columns
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

importance_df

,Feature,Importance
1,Glucose,0.239169
5,BMI,0.155007
4,Insulin,0.131713
7,Age,0.123472
6,DiabetesPedigreeFunction,0.107954
3,SkinThickness,0.091856
0,Pregnancies,0.080156
2,BloodPressure,0.070673


In [40]:
importance_df.to_csv('feature_importance.csv', index=False)

In [41]:
y_pred_rf_bal = pipeline_rf_bal.predict(X_test)

print(f"Classification Report:\n{classification_report(y_test, y_pred_rf_bal)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.67      0.74        95
           1       0.54      0.74      0.63        50

    accuracy                           0.70       145
   macro avg       0.69      0.71      0.69       145
weighted avg       0.73      0.70      0.70       145



"Comparision of Classification Report".

In [43]:
print(".....................LR...................")

print(f"Classification Report:\n{classification_report(y_test, y_pred_lr)}")
print(".................LR Balanced..................")

print(f"Classification Report:\n{classification_report(y_test, y_pred_lr_bal)}")
print("..........................RF.....................")

print(f"Classification Report:\n{classification_report(y_test, y_pred_rf)}")
print("......................RF Balanced.................")

print(f"Classification Report:\n{classification_report(y_test, y_pred_rf_bal)}")

.....................LR...................
Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.80      0.78        95
           1       0.58      0.52      0.55        50

    accuracy                           0.70       145
   macro avg       0.67      0.66      0.66       145
weighted avg       0.70      0.70      0.70       145

.................LR Balanced..................
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.69      0.74        95
           1       0.53      0.66      0.59        50

    accuracy                           0.68       145
   macro avg       0.66      0.68      0.67       145
weighted avg       0.70      0.68      0.69       145

..........................RF.....................
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.81      0.81        95
           1       0.64     

# Final Model: Save Pipeline

In [44]:
import joblib
joblib.dump(pipeline_rf_bal, 'diabetes_model.pkl')

['diabetes_model.pkl']

In [45]:
loaded_pipeline = joblib.load('diabetes_model.pkl')
sample_prediction = loaded_pipeline.predict(X_test.iloc[[0]])
print(sample_prediction)

[1]
